# 08 RAG/Agent 生产化加固：把应用经验升级为 Infra 能力

![RAG and Agent hardening](../assets/images/rag_agent_hardening.png)

你已有 Agent/RAG 构建经验，本章把这些经验升级成生产 Infra 能力。核心观点：RAG/Agent 的生产问题，很多不是“模型不够聪明”，而是数据、状态、工具、安全、评测和观测没有工程化。


## 1. Production RAG/Agent 拓扑

```mermaid
flowchart TB
    User["用户 / User"] --> Orchestrator["Agent/RAG 编排器 / Orchestrator"]
    Docs["文档 / Documents"] --> Parser["解析与清洗 / parse / clean"]
    Parser --> Chunk["切块与版本 / chunk / version"]
    Chunk --> Emb["嵌入流水线 / embedding pipeline"]
    Emb --> VDB["向量数据库 / Vector DB"]
    Orchestrator --> Retrieve["检索器 / retriever"]
    Retrieve --> VDB
    Retrieve --> Rerank["重排器 / reranker"]
    Rerank --> Prompt["提示词组装 / prompt assembly"]
    Prompt --> LLM["LLM 推理服务 / LLM serving"]
    Orchestrator --> Tools["工具沙箱 / tool sandbox"]
    Tools --> Queue["队列与重试 / queue / retry"]
    Orchestrator --> State["状态存储 / state store"]
    Orchestrator --> Obs["追踪/指标/日志 / trace / metrics / logs"]
    Orchestrator --> Eval["评测与反馈 / eval / feedback"]
    Human["人工审批 / human approval"] --> Orchestrator
```

这张图里，LLM 只是其中一层。很多生产事故发生在文档解析、权限过滤、检索、工具调用、状态恢复和评测缺失。


## 2. RAG 数据管道

RAG 的质量首先取决于数据。生产级数据管道至少要处理：

- **解析**：PDF、HTML、表格、图片 OCR、代码块、标题层级。
- **清洗**：去页眉页脚、去重复、保留结构、处理乱码。
- **chunking**：按语义、标题、长度、重叠策略切分。
- **metadata**：文档 id、版本、权限、时间、来源、章节、hash。
- **embedding refresh**：文档更新后哪些 chunk 需要重算 embedding。
- **回滚**：错误文档入库后能否撤销。

RAG demo 常常忽略版本。一旦用户问“为什么昨天答对今天答错”，没有版本就无法定位。


In [ ]:
# chunk versioning 示例：让每个 chunk 可追踪来源和版本。
import hashlib

def chunk_doc(doc_id, version, text, chunk_size=32):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        h = hashlib.sha1(f'{doc_id}:{version}:{i}:{chunk}'.encode()).hexdigest()[:10]
        chunks.append({
            'chunk_id': f'{doc_id}:v{version}:{i//chunk_size}:{h}',
            'doc_id': doc_id,
            'version': version,
            'offset': i,
            'text': chunk,
        })
    return chunks

v1 = chunk_doc('policy', 1, '报销制度要求发票真实、审批完整、金额与预算一致。')
v2 = chunk_doc('policy', 2, '报销制度要求发票真实、审批完整、金额与预算一致，差旅需提前申请。')
print(v1)
print(v2)


## 3. 检索、rerank 与上下文压缩

常见 RAG 链路：query rewrite → hybrid search → rerank → context compression → prompt assembly。

- **query rewrite**：把用户问题改写成更适合检索的查询。
- **hybrid search**：结合向量检索和关键词检索，减少单一路径漏召回。
- **rerank**：用更强模型对候选段落重新排序。
- **context compression**：把长文档压缩成模型真正需要的证据，减少 prompt token。

不要把所有召回内容都塞进 prompt。长 prompt 会增加 prefill 成本、KV cache 和 hallucination 风险。


## 4. Agent 工具调用加固

Agent 最大风险在于它会调用工具，工具可能有副作用。生产系统必须处理：

| 风险 | 加固方式 |
|---|---|
| 工具超时 | timeout、retry、fallback |
| 重复执行 | 幂等 key、操作日志 |
| 高风险动作 | human approval、policy check |
| 越权访问 | scoped credential、权限过滤 |
| prompt injection | 工具说明隔离、输入净化、策略模型 |
| 状态丢失 | state store、checkpoint、resume |


In [ ]:
# 工具调用加固：timeout/retry/fallback 的最小模型。
import random

random.seed(7)

def call_tool_once():
    r = random.random()
    if r < 0.2:
        raise TimeoutError('tool timeout')
    if r < 0.35:
        raise RuntimeError('transient error')
    return {'ok': True, 'value': 42}

def call_tool(max_attempts=3):
    errors = []
    for attempt in range(1, max_attempts + 1):
        try:
            return call_tool_once()
        except Exception as exc:
            errors.append(f'attempt {attempt}: {exc}')
    return {'ok': False, 'fallback': 'ask_human_or_degrade', 'errors': errors}

for _ in range(5):
    print(call_tool())


## 5. 评测与观测

RAG/Agent 不能只看最终答案。你需要分层指标：

- 检索层：gold doc recall、MRR、rerank 命中率、无权限文档泄露率。
- Prompt 层：prompt token 数、上下文利用率、模板版本。
- 模型层：答案正确率、引用一致性、拒答准确率、格式成功率。
- 工具层：tool success rate、timeout rate、retry count、approval rate。
- 用户层：满意度、人工纠错、升级人工率。

上线前做 offline eval，上线后做 canary 和线上抽样评测。每次改 chunk 策略、embedding 模型、reranker、prompt 或模型，都要跑回归。


## 6. 从 demo 到 production 的迁移案例

假设你已有一个能回答制度问题的 RAG demo。迁移路线：

1. 加文档版本和权限 metadata。
2. 建立小型 golden set：问题、标准答案、证据文档、不能回答的边界问题。
3. 给检索链路加 trace：query、top-k、rerank score、最终 context。
4. 给 prompt 和模型加版本号。
5. 给模型调用接入网关：限流、预算、fallback、日志。
6. 加线上反馈入口：用户标错、人工修正、自动进入 eval backlog。
7. 对工具调用加 sandbox、timeout、retry 和 human approval。
8. 建 dashboard：latency、cost、retrieval hit、tool failure、answer quality sample。

这样你讲 RAG/Agent 项目时，就不只是“我会搭链路”，而是“我知道如何把它变成可治理系统”。


## 7. 常见误区

- **只优化最终回答，不看检索证据。** 回答对了也可能是碰巧，必须看证据链。
- **所有文档同权。** 生产中权限、时效、来源可信度都很重要。
- **工具调用没有审计。** 高风险工具必须可追踪、可回滚、可审批。
- **没有回归集。** 每次改 chunk/prompt/model 都可能让旧问题退化。
- **把 token 成本当小事。** RAG prompt 很容易因为上下文膨胀造成成本失控。


## 8. Prompt injection 与工具安全

RAG/Agent 特别容易遇到 prompt injection：恶意文档或用户输入试图覆盖系统指令、诱导泄露数据、调用高风险工具。防护不是一句“请忽略恶意指令”能解决的，需要多层策略：

- 检索文档只作为不可信上下文，不允许覆盖 system policy。
- 工具调用前做 policy check，高风险动作需要 human approval。
- 工具凭证最小权限，不把长期高权限 token 暴露给模型。
- 对外部网页、用户上传文档、邮件等来源做风险标记。
- 输出前做敏感信息和越权内容检查。
- trace 中保留文档来源和工具调用决策，便于审计。

Agent 的核心风险不是“答错”，而是“带着权限做错事”。


## 9. RAG/Agent failure mode 清单

| Failure mode | 表现 | 应对 |
|---|---|---|
| 检索漏召回 | 正确文档没进 top-k | hybrid search、query rewrite、golden set recall |
| rerank 错排 | 正确文档在候选里但没被选中 | reranker 评测、特征分析、阈值调整 |
| 上下文过长 | TTFT 高、成本高、答案跑偏 | context compression、摘要、限制 token |
| 数据过期 | 答案引用旧政策 | 版本、更新时间、回滚和刷新任务 |
| 工具超时 | Agent 卡住或重复调用 | timeout、retry、circuit breaker |
| 工具越权 | 执行了不该执行的动作 | policy、scoped credential、human approval |
| 无法复盘 | 用户投诉但找不到原因 | request id、trace、prompt/data/model version |


## 10. 面试/工作表达

> 我做 RAG/Agent 不会只停留在 prompt 和链路 demo。我会把它拆成数据管道、检索质量、prompt 版本、推理服务、工具安全、状态管理、观测和评测。比如回答错误时，我会先看检索是否命中证据，再看 rerank、prompt、模型输出和工具调用；上线前会用 golden set 做回归，上线后用 trace 和用户反馈持续迭代。


## 11. 数据权限是 RAG 的生产红线

企业 RAG 最大风险之一是越权检索。即使模型回答本身没有恶意，只要检索阶段把用户无权访问的文档塞进 prompt，就已经发生泄露。生产 RAG 必须在检索前或检索时做权限过滤，而不是指望模型“不要说”。

常见做法：chunk metadata 中带 ACL、department、confidentiality、effective_time；检索时把用户身份和权限转成 filter；返回答案时只引用用户有权访问的文档；trace 中保留权限过滤结果用于审计。


## 12. Agent 状态管理

Agent 常常是多步过程：计划、检索、调用工具、观察结果、继续推理。生产中必须管理状态：当前任务到哪一步、已调用哪些工具、是否可以重试、用户是否批准、失败后能否恢复。

如果没有状态管理，服务重启或工具超时可能导致重复扣款、重复发邮件、重复写数据库。工程上常用 state store、checkpoint、idempotency key、任务队列和 human approval，把 Agent 从“单次脚本”变成“可恢复工作流”。


## 13. 自测题

1. 为什么 RAG 权限过滤不能只放在生成后？
2. chunk versioning 解决什么问题？
3. rerank 指标应该如何和最终答案指标结合？
4. Agent 工具调用为什么需要幂等和审计？
5. 从 demo 迁移到 production，你会优先补哪三项基础设施？


## 14. 把 RAG/Agent 项目讲成 Infra 项目

如果你要把已有项目包装成“懂 AI Infra”的经历，可以按生产化改造讲。不要只说用了哪个向量库、哪个 Agent 框架，而要讲你如何处理生产问题：数据如何更新，检索如何评测，模型调用如何观测，工具如何安全执行，失败如何降级。

一个好的项目叙述可以是：我先搭建了基础 RAG 链路，然后发现长上下文导致 TTFT 和成本上升，于是加入 rerank 和 context compression；发现文档更新后无法复盘，于是给 chunk 加 doc_id/version/hash；发现线上问题难定位，于是给每次请求记录 retrieval docs、prompt version、model version、token usage 和 latency breakdown；发现工具调用有副作用，于是增加 timeout、retry、idempotency key 和 human approval。这样讲，面试官会看到你不是只会拼组件，而是理解 AI 系统上线后的工程约束。


## 来源与延伸阅读

本章正文已经把关键内容消化进教程，下面链接只用于延伸阅读和核对最新细节：vLLM、vLLM-Metal、Ray Serve LLM、SGLang、TensorRT-LLM、Text Generation Inference、LiteLLM、KServe、NVIDIA GPU Operator、OpenTelemetry GenAI。
